# Creating synthetic observations for Galaxy Populations


In [ ]:
from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import z_at_value
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from synthesizer.grid import Grid
from synthpop import GalaxyPopulation
from synthpop.models import Model, Default, Constant
from unyt import yr, Myr, Msun, Gyr, unyt_quantity, Mpc, dimensionless


In [ ]:
# load grid
grid = Grid("test_grid")

# load default model
model = Default()
model = Constant()

# Define the volume of the galaxy population
volume = 1E5 * Mpc**3

In [ ]:

# Instantiate the galaxy population
galpop = GalaxyPopulation(
    minimum_stellar_mass=1E9*Msun, 
    maximum_stellar_mass=1E11*Msun, 
    volume=volume,
    model=model,
    grid=grid,
    cosmology=cosmo,
    redshift=0.0,
    random_seed=42)

print(galpop)




In [ ]:
from synthesizer.emission_models import IncidentEmission, EmergentEmission

from synthesizer.emission_models.attenuation import PowerLaw

incident = IncidentEmission(grid=grid)


emergent = EmergentEmission(
    grid=grid,
    dust_curve=PowerLaw(slope=-1),
    fesc=1E-10,)
print(emergent)


## Spectra

Before generating other photometry we need to generate spectra.

In [ ]:
# galpop.generate_spectra(incident)
galpop.generate_spectra(emergent)

print(galpop.galaxies[0].stars.spectra.keys())

galpop.plot_spectra('emergent')

## Photometry

To generate broadband photometry we need to provide a `synthesizer.filters.FilterSet` object. 

In [ ]:

from synthesizer.filters import UVJ

# Get a UVJ filter collection
uvj = UVJ(new_lam=grid.lam)

galpop.generate_photometry('incident', uvj)
galpop.generate_photometry('emergent', uvj)

In [ ]:
emergent_V = galpop.get_photometry('emergent', 'V')
incident_V = galpop.get_photometry('incident', 'V')
attenuation = -2.5*np.log10(emergent_V/incident_V)

plt.scatter(galpop.surviving_masses.to("Msun"), attenuation, alpha=0.5)
plt.xlabel("$M_{\star}/M_{\odot}$")
plt.ylabel("$A_V$")
plt.xscale("log")
plt.title("Attenuation vs. Surviving Mass")
plt.show()


### Luminosity functions

In [ ]:
galpop.plot_luminosity_function('incident', 'U')
galpop.plot_luminosity_function('emergent', 'U')

### Colour diagrams

Methods exist for rapidly producing colour-magnitude and colour-colour diagrams.

In [ ]:
galpop.plot_color_color_diagram('emergent', ['U','V','J'])